In [1]:
# SparkSession
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, when, count

spark = SparkSession.builder \
    .appName("Tugas4") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("SparkSession berhasil dibuat!")
print("Versi Spark:", spark.version)

26/09/10 17:33:08 WARN Utils: Your hostname, yan resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/09/10 17:33:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/10 17:33:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession berhasil dibuat!
Versi Spark: 3.5.9


In [2]:
# A. Membaca dan Eksplorasi Awal
# Baca dataset dari HDFS, tampilkan printSchema(), jumlah baris (count()), dan 10 baris pertama (show(10)).
df = spark.read.csv(
    "hdfs://localhost:9000/user/yan/tugas4/transaksi_september_2026.csv",
    header=True, inferSchema=True
)

df.printSchema()
print("Jumlah baris:", df.count())
df.show(10)

root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Jumlah baris: 1000
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|        

In [3]:
# B. Menangani Data Kosong
# Kolom rating memiliki nilai kosong. Tampilkan berapa banyak
jumlah_kosong = df.filter(col("rating").isNull()).count()
print("Jumlah rating kosong:", jumlah_kosong)

df = df.na.drop(subset=["rating"])

print("Jumlah baris setelah baris kosong dihapus:", df.count())

Jumlah rating kosong: 204
Jumlah baris setelah baris kosong dihapus: 796


Gunakan df.na.fill() atau df.na.drop() (pilih salah satu, jelaskan alasannya pada markdown cell) untuk menanganinya.

Baris yang ratingnya kosong dihapus, jika di isi maka data itu bukan rating sungguhan dan bisa bikin hasil analisis rating jadi kurang akurat.

In [4]:
# C. Transformasi Data
# Tambahkan kolom total_pendapatan (unit_terjual x harga_satuan), lalu tambahkan kolom tier_transaksi yang bernilai "Besar" jika total_pendapatan > 500000, atau "Kecil" jika sebaliknya 
df = df.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))

df = df.withColumn(
    "tier_transaksi",
    when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil")
)

df.select("order_id", "unit_terjual", "harga_satuan", "total_pendapatan", "tier_transaksi").show(10)

+--------+------------+------------+----------------+--------------+
|order_id|unit_terjual|harga_satuan|total_pendapatan|tier_transaksi|
+--------+------------+------------+----------------+--------------+
|ORD-3000|           3|       90000|          270000|         Kecil|
|ORD-3001|           3|      200000|          600000|         Besar|
|ORD-3002|           8|       60000|          480000|         Kecil|
|ORD-3003|           6|      350000|         2100000|         Besar|
|ORD-3004|          10|       60000|          600000|         Besar|
|ORD-3005|           5|       20000|          100000|         Kecil|
|ORD-3006|           2|       20000|           40000|         Kecil|
|ORD-3008|           7|       20000|          140000|         Kecil|
|ORD-3009|          10|       90000|          900000|         Besar|
|ORD-3010|           4|      350000|         1400000|         Besar|
+--------+------------+------------+----------------+--------------+
only showing top 10 rows



In [5]:
# D. Analisis dengan GroupBy
# 1. Kategori apa yang memiliki total_pendapatan tertinggi?
print("No.1. Kategori dengan pendapatan tertinggi:")
df.groupBy("kategori").agg(
    spark_sum("total_pendapatan").alias("total_pendapatan")
).orderBy(col("total_pendapatan").desc()).show(1)

# 2. Kota mana dengan jumlah transaksi tier "Besar" terbanyak?
print("No.2. Kota dengan transaksi tier Besar terbanyak:")
df.filter(col("tier_transaksi") == "Besar") \
  .groupBy("kota") \
  .count() \
  .orderBy(col("count").desc()) \
  .show(1)

# 3. Berapa rata-rata rating untuk masing-masing metode_pembayaran (data kosong sudah ditangani di bagian B)?
print("No.3. Rata-rata rating per metode pembayaran:")
df.groupBy("metode_pembayaran").agg(
    avg("rating").alias("rata_rata_rating")
).orderBy(col("rata_rata_rating").desc()).show()

No.1. Kategori dengan pendapatan tertinggi:
+------------+----------------+
|    kategori|total_pendapatan|
+------------+----------------+
|Rumah Tangga|       108285000|
+------------+----------------+
only showing top 1 row

No.2. Kota dengan transaksi tier Besar terbanyak:
+----+-----+
|kota|count|
+----+-----+
|Solo|   74|
+----+-----+
only showing top 1 row

No.3. Rata-rata rating per metode pembayaran:
+-----------------+-----------------+
|metode_pembayaran| rata_rata_rating|
+-----------------+-----------------+
|              COD|4.172413793103448|
|    Transfer Bank| 4.16256157635468|
|         E-Wallet|4.135678391959799|
|     Kartu Kredit|4.109947643979058|
+-----------------+-----------------+



In [6]:
# E. Menyimpan Hasil ke HDFS
# Simpan DataFrame hasil olahan bagian C (lengkap dengan kolom total_pendapatan dan tier_transaksi) ke HDFS dalam format CSV baru, kemudian verifikasi apakah sudah berhasil.
df.write.csv(
    "hdfs://localhost:9000/user/yan/tugas4/hasil_olahan",
    header=True,
    mode="overwrite"
)
print("Berhasil disimpan ke HDFS.")

Berhasil disimpan ke HDFS.


In [7]:
# verifikasi apakah sudah berhasil
!hdfs dfs -ls /user/yan/tugas4/hasil_olahan

Found 2 items
-rw-r--r--   3 yan supergroup          0 2026-09-10 17:33 /user/yan/tugas4/hasil_olahan/_SUCCESS
-rw-r--r--   3 yan supergroup      73357 2026-09-10 17:33 /user/yan/tugas4/hasil_olahan/part-00000-6dd0e84c-7f1b-4dfa-99f3-c15e6f9a1852-c000.csv


Hasilnya kesimpan jadi banyak file kecil, bukan satu file. Karena Spark memproses data secara paralel di beberapa "pekerja", dan tiap pekerja nyimpen hasilnya sendiri-sendiri, makanya jadi banyak file.